# THP vs RoTHP vs HoTHP — Extrapolacao em Dados Reais

**Como rodar:** Runtime -> Change runtime type -> T4 GPU

---

## O que este notebook prova?

Treinamos THP, RoTHP e HoTHP em **prefixos curtos** (10 eventos) das sequencias reais
e depois testamos nas **sequencias completas**, medindo a NLL total.

Isso eh o cenario de **extrapolacao de horizonte**: o modelo so viu 10 eventos no treino,
mas precisa modelar sequencias muito mais longas no teste.

### Datasets

Testamos **todos** os datasets do EasyTPP para nao desperdicar oportunidades:

| Dataset | Tipos | Seq len media | Fator de extrap (~) |
|---|---|---|---|
| taobao | 17 | 56 | 5-6x |
| amazon | 16 | 45 | 4-9x |
| retweet | 3 | 109 | ~10x |
| taxi | 10 | 37 | ~3x |
| stackoverflow | 22 | 65 | ~6x |
| earthquake | 7 | 17 | ~1.7x |
| volcano | 1 | 15 | ~1.5x |

### Protocolo

1. **Treino:** sequencias truncadas nos primeiros 10 eventos
2. **Teste:** NLL **completa** nas sequencias inteiras (sem mascara ad-hoc)
3. Metrica principal: NLL total na sequencia completa (menor = melhor)

**Tempo estimado:** ~1-2h numa GPU T4 (7 datasets x 5 seeds x 3 modelos)

In [ ]:
# ============================================================================
# CELULA 1 — Instalacao e clone do repositorio
# ============================================================================

import os

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

!pip install omegaconf datasets -q

# ── Fix 1: model/__init__.py importa todos os modelos (ODETPP, S2P2, etc.)
# e alguns falham por dependencias ausentes. Reescrevemos com imports minimos.
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write("""from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP as TorchTHP
from easy_tpp.model.torch_model.torch_rothp import RoTHP as TorchRoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP as TorchHoTHP
""")

# ── Fix 2: torch_hothp.py importa attention_fixed de um notebook (erro).
# Removemos essa linha — o patch de attention eh feito na celula 2.
_hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
with open(_hothp_path, 'r') as f:
    code = f.read()
if 'from notebooks.' in code:
    code = code.replace(
        'from notebooks.Extrapolation_and_Attention_Analysis import attention_fixed\n', '')
    with open(_hothp_path, 'w') as f:
        f.write(code)
    print('Fix: removido import quebrado de notebooks em torch_hothp.py')

print('Instalacao OK')

In [ ]:
# ============================================================================
# CELULA 2 — Imports, configuracoes e reproducibilidade
# ============================================================================

import os, sys, math, random, hashlib, contextlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Parametros do experimento ──────────────────────────────────────────────

DATASET_NAMES = [
    'taobao', 'amazon', 'retweet', 'taxi',
    'stackoverflow', 'earthquake', 'volcano',
]
TRAIN_LEN     = 10           # treina so nos primeiros 10 eventos de cada sequencia
N_SEEDS       = 5
EPOCHS        = 100
PATIENCE      = 20
BASE_SEED     = 42
BATCH_SIZE    = 256
EVAL_BS       = 256

# Cores dos modelos
COR = {'THP': '#999999', 'RoTHP': '#4C72B0', 'HoTHP': '#C44E52'}

# ── Reproducibilidade ─────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)

# ── Dispositivo e AMP ─────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'

if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

# ── Carrega modelos ───────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention

import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

MODEL_CLASSES = [
    ('THP',   THP,   1e-3),
    ('RoTHP', RoTHP, 1e-3),
    ('HoTHP', HoTHP, 5e-4),
]

print(f'Dispositivo: {device}  |  AMP: {USE_AMP}')
print(f'TRAIN_LEN: {TRAIN_LEN} eventos (treino truncado)')
print(f'Datasets: {DATASET_NAMES}')
print(f'Modelos: {[name for name, _, _ in MODEL_CLASSES]}')
print(f'Seeds: {N_SEEDS}')

In [ ]:
# ============================================================================
# CELULA 3 — Download dos datasets e preparacao dos dados
#
# Estrategia de extrapolacao com dados reais:
#   - TREINO: sequencias truncadas nos primeiros TRAIN_LEN eventos
#   - TESTE: sequencias completas (usamos as que tem > TRAIN_LEN eventos)
#   - Avaliamos NLL in-dist (posicoes < TRAIN_LEN) e OOD (posicoes >= TRAIN_LEN)
# ============================================================================

def collate_fn(batch_list, pad_id, time_scale):
    """Collate para dados reais do HuggingFace.
    Normaliza tempos pelo gap medio do treino."""
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)

    pad_time  = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type  = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    npm       = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attn      = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal    = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)

    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)

        ts = (ts - ts[0]) / time_scale
        td = td / time_scale

        pad_time[i, :l]  = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l]  = ev
        npm[i, :l]       = 1.0

        m = causal.clone()
        m[:, l:] = True
        m[l:, :] = True
        attn[i]  = m

    return pad_time, pad_delta, pad_type, npm, attn


def collate_fn_truncated(batch_list, pad_id, time_scale, max_events):
    """Collate que trunca cada sequencia nos primeiros max_events eventos.
    Usado para o treino: o modelo so ve prefixos curtos."""
    truncated = []
    for item in batch_list:
        truncated.append({
            'time_since_start':      item['time_since_start'][:max_events],
            'time_since_last_event': item['time_since_last_event'][:max_events],
            'type_event':            item['type_event'][:max_events],
        })
    return collate_fn(truncated, pad_id, time_scale)


# Download e pre-processamento
raw_datasets = {}
dataset_info = {}

for ds_name in DATASET_NAMES:
    print(f'Baixando easytpp/{ds_name}...')
    ds = load_dataset(f'easytpp/{ds_name}')
    raw_datasets[ds_name] = ds

    # time_scale a partir dos deltas do treino (antes de truncar)
    all_deltas = []
    for item in ds['train']:
        all_deltas.extend([d for d in item['time_since_last_event'] if d > 0])
    time_scale = np.mean(all_deltas)

    # Numero de tipos
    max_type = 0
    for x in ds['train']:
        if len(x['type_event']) > 0:
            max_type = max(max_type, max(x['type_event']))
    num_types = max_type + 1
    pad_id = num_types

    # Estatisticas de comprimento
    train_lens = [len(x['time_since_start']) for x in ds['train']]
    test_lens  = [len(x['time_since_start']) for x in ds['test']]

    # Filtra: para teste OOD, so sequencias com > TRAIN_LEN eventos
    test_long_indices = [i for i, l in enumerate(test_lens) if l > TRAIN_LEN]
    test_long = ds['test'].select(test_long_indices)

    dataset_info[ds_name] = {
        'time_scale': time_scale,
        'num_types': num_types,
        'pad_id': pad_id,
        'test_long': test_long,
    }

    n_ood = len(test_long_indices)
    total_test = len(test_lens)
    mean_test_len = np.mean([test_lens[i] for i in test_long_indices]) if test_long_indices else 0

    print(f'  types={num_types}  time_scale={time_scale:.4f}')
    print(f'  train: {len(train_lens)} seqs, len media={np.mean(train_lens):.1f}, '
          f'min={np.min(train_lens)}, max={np.max(train_lens)}')
    print(f'  test:  {total_test} seqs, len media={np.mean(test_lens):.1f}, '
          f'min={np.min(test_lens)}, max={np.max(test_lens)}')
    print(f'  test OOD (len > {TRAIN_LEN}): {n_ood}/{total_test} seqs '
          f'({100*n_ood/total_test:.0f}%), len media={mean_test_len:.1f}')
    if n_ood > 0:
        ood_lens = [test_lens[i] for i in test_long_indices]
        total_events = sum(ood_lens)
        ood_events = sum(max(0, l - TRAIN_LEN) for l in ood_lens)
        print(f'  eventos OOD: {ood_events}/{total_events} '
              f'({100*ood_events/total_events:.0f}%) estao alem da posicao {TRAIN_LEN}')
        extrap_factor = mean_test_len / TRAIN_LEN
        print(f'  fator de extrapolacao medio: {extrap_factor:.1f}x')
    print()

In [ ]:
# ============================================================================
# CELULA 4 — Funcoes de treino e avaliacao OOD
# ============================================================================

def make_config(num_types, pad_id):
    return ModelConfig(**{
        'hidden_size':   64,
        'num_layers':    2,
        'num_heads':     4,
        'dropout_rate':  0.1,
        'num_event_types':     num_types,
        'num_event_types_pad': num_types + 1,
        'event_pad_index':     pad_id,
        'time_emb_size': 64,
        'use_ln': True,
        'gpu': 0 if torch.cuda.is_available() else -1,
        'model_id': 'Benchmark',
        'thinning': {
            'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
            'patience_counter': 5, 'num_samples_boundary': 5,
            'dtime_max': 5.0, 'num_step_gen': 1,
        },
        'loss_integral_num_sample_per_step': 20,
        'use_mc_samples': False,
    })


def eval_nll(model, dl):
    """NLL media sobre todos os eventos."""
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def eval_indist_nll(model, dl):
    """NLL apenas nas posicoes < TRAIN_LEN (in-distribution)."""
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            mask = npm.clone()
            mask[:, TRAIN_LEN:] = 0.0   # zera posicoes OOD
            if mask.sum() == 0:
                continue
            with _autocast():
                l, n = model.loglike_loss([t, d, k, mask, attn])
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def eval_ood_nll(model, dl):
    """NLL apenas nas posicoes >= TRAIN_LEN (fora da distribuicao de treino)."""
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            mask = npm.clone()
            mask[:, :TRAIN_LEN] = 0.0   # zera posicoes in-dist
            if mask.sum() == 0:
                continue
            with _autocast():
                l, n = model.loglike_loss([t, d, k, mask, attn])
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def train_model(cls, config, train_dl, val_dl, lr, base_seed):
    set_seed(base_seed)
    m = cls(config).to(device)

    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=8, min_lr=1e-5)
    scaler = _Scaler(enabled=True) if USE_AMP else None

    best_val   = float('inf')
    best_state = None
    no_imp     = 0

    for ep in range(EPOCHS):
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = m.loglike_loss(batch)
                nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()

        v = eval_nll(m, val_dl)
        sched.step(v)

        if v < best_val - 1e-4:
            best_val   = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= PATIENCE:
            break

    m.load_state_dict(best_state)
    return m, best_val


print('Funcoes prontas.')

In [ ]:
# ============================================================================
# CELULA 5 — Execucao principal
#
# Para cada dataset x seed x modelo:
#   1. Treina com sequencias truncadas (primeiros TRAIN_LEN eventos)
#   2. Avalia NLL COMPLETA nas sequencias longas inteiras
#      (metrica principal — sem mascara ad-hoc)
#   3. Tambem reporta NLL truncada (in-dist) para comparacao
#
# Tempo estimado: ~20-40 min numa T4
# ============================================================================

seeds = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
all_results = []

for ds_name in DATASET_NAMES:
    info = dataset_info[ds_name]
    ds   = raw_datasets[ds_name]

    print(f'\n{"="*60}')
    print(f'  DATASET: {ds_name}')
    print(f'  Treino: prefixos de {TRAIN_LEN} eventos')
    print(f'  Teste: NLL completa nas sequencias inteiras')
    print(f'{"="*60}')

    config = make_config(info['num_types'], info['pad_id'])

    # Collate para treino: trunca nos primeiros TRAIN_LEN eventos
    _collate_train = lambda batch, _pid=info['pad_id'], _ts=info['time_scale']: \
        collate_fn_truncated(batch, _pid, _ts, TRAIN_LEN)

    # Collate para teste: sequencias completas
    _collate_full = lambda batch, _pid=info['pad_id'], _ts=info['time_scale']: \
        collate_fn(batch, _pid, _ts)

    # Validacao tambem truncada (mesma distribuicao do treino)
    val_dl = DataLoader(ds['validation'], batch_size=EVAL_BS,
                        shuffle=False, collate_fn=_collate_train)

    # Teste in-dist: sequencias truncadas (mesma distribuicao)
    test_short_dl = DataLoader(info['test_long'], batch_size=EVAL_BS,
                               shuffle=False, collate_fn=_collate_train)

    # Teste extrapolacao: sequencias COMPLETAS (5-9x mais longas que o treino)
    test_full_dl = DataLoader(info['test_long'], batch_size=EVAL_BS,
                              shuffle=False, collate_fn=_collate_full)

    for seed_idx, seed in enumerate(seeds):
        print(f'\n  Seed {seed_idx+1}/{N_SEEDS} (seed={seed})')

        g = torch.Generator()
        g.manual_seed(run_seed(ds_name, seed))
        train_dl = DataLoader(ds['train'], batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=_collate_train, generator=g)

        for model_name, model_cls, lr in MODEL_CLASSES:
            t0 = time.time()
            model, val_nll = train_model(
                model_cls, config, train_dl, val_dl,
                lr=lr,
                base_seed=run_seed(ds_name, model_name, seed),
            )
            train_time = time.time() - t0

            # Metrica principal: NLL completa nas sequencias longas
            nll_full  = eval_nll(model, test_full_dl)

            # Diagnostico: NLL truncada (mesma distribuicao do treino)
            nll_short = eval_nll(model, test_short_dl)

            all_results.append({
                'dataset':    ds_name,
                'model':      model_name,
                'seed':       seed,
                'val_nll':    val_nll,
                'nll_short':  nll_short,    # NLL in-dist (truncada)
                'nll_full':   nll_full,     # NLL extrapolacao (completa)
                'train_time': train_time,
            })

            print(f'    {model_name:6s}  val={val_nll:.4f}  '
                  f'short={nll_short:.4f}  full={nll_full:.4f}  ({train_time:.0f}s)')

df = pd.DataFrame(all_results)
print('\nExperimento concluido!')

In [ ]:
# ============================================================================
# CELULA 6 — Tabela resumo e testes de significancia
# ============================================================================

model_names = ['THP', 'RoTHP', 'HoTHP']

print('=' * 95)
print(f'RESULTADOS DE EXTRAPOLACAO  (treino: {TRAIN_LEN} eventos, teste: sequencias completas)')
print(f'n = {N_SEEDS} seeds')
print('=' * 95)

for ds_name in DATASET_NAMES:
    sub = df[df['dataset'] == ds_name]

    # Calcula fator de extrapolacao medio
    test_lens = [len(x['time_since_start']) for x in dataset_info[ds_name]['test_long']]
    mean_extrap = np.mean(test_lens) / TRAIN_LEN

    print(f'\n  {ds_name.upper()}  (fator de extrapolacao medio: {mean_extrap:.1f}x)')
    print(f'  {"Model":>6s}  {"NLL short":>14s}  {"NLL full":>14s}')
    print(f'  {"-"*6}  {"-"*14}  {"-"*14}')

    for mname in model_names:
        rows = sub[sub['model'] == mname]
        print(f'  {mname:>6s}  '
              f'{rows["nll_short"].mean():.4f}+/-{rows["nll_short"].std():.4f}  '
              f'{rows["nll_full"].mean():.4f}+/-{rows["nll_full"].std():.4f}')

    print()
    print(f'  NLL short = in-dist ({TRAIN_LEN} eventos, mesma distribuicao do treino)')
    print(f'  NLL full  = extrapolacao (sequencia completa ~{np.mean(test_lens):.0f} eventos)')

    # Testes de significancia sobre NLL full (metrica principal)
    ho = sub[sub['model'] == 'HoTHP']['nll_full'].values
    for baseline in ['RoTHP', 'THP']:
        bl = sub[sub['model'] == baseline]['nll_full'].values
        if len(ho) >= 2 and len(bl) >= 2:
            t_stat, p2 = stats.ttest_rel(bl, ho)
            # positivo t_stat = baseline pior que HoTHP (HoTHP melhor)
            p1 = p2 / 2 if t_stat > 0 else 1.0
            sig = '***' if p1 < 0.001 else '**' if p1 < 0.01 else '*' if p1 < 0.05 else '~' if p1 < 0.10 else 'ns'
            diff = bl.mean() - ho.mean()
            print(f'    HoTHP vs {baseline} (NLL full): diff={diff:+.4f}  p={p1:.4f} {sig}')

In [ ]:
# ============================================================================
# CELULA 7 — Grafico principal: NLL short vs NLL full (extrapolacao)
#
# Barras claras = NLL in-dist (sequencias truncadas, mesma distrib do treino)
# Barras escuras = NLL completa (sequencias inteiras, cenario de extrapolacao)
#
# Se a barra escura eh muito maior que a clara, o modelo degradou ao extrapolar.
# ============================================================================

n_ds = len(DATASET_NAMES)
fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 6))
if n_ds == 1:
    axes = [axes]

for ax, ds_name in zip(axes, DATASET_NAMES):
    sub = df[df['dataset'] == ds_name]
    test_lens = [len(x['time_since_start']) for x in dataset_info[ds_name]['test_long']]
    mean_extrap = np.mean(test_lens) / TRAIN_LEN

    x = np.arange(len(model_names))
    w = 0.3

    for offset, metric, label, alpha in [
        (-w/2, 'nll_short', f'In-dist ({TRAIN_LEN} eventos)', 0.45),
        (+w/2, 'nll_full',  f'Extrapolacao ({mean_extrap:.0f}x)', 0.9),
    ]:
        means = []
        stds  = []
        colors = []
        for mname in model_names:
            vals = sub[sub['model'] == mname][metric].values
            means.append(vals.mean())
            stds.append(vals.std())
            colors.append(COR[mname])
        bars = ax.bar(x + offset, means, w, yerr=stds, capsize=4,
                      color=colors, alpha=alpha, edgecolor='white', linewidth=1,
                      label=label, error_kw={'lw': 1})

        # Valor numerico
        for bar, m, s in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + s + 0.01,
                    f'{m:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=13)
    ax.set_ylabel('NLL (nats)', fontsize=12)
    ax.set_title(f'{ds_name}', fontweight='bold', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f'NLL In-dist vs Extrapolacao — Dados Reais\n'
    f'Treino: {TRAIN_LEN} eventos  |  n = {N_SEEDS} seeds',
    fontsize=13, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.savefig('extrapolation_nll_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 8 — Grafico de dispersao por seed (NLL full)
# ============================================================================

fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 5))
if n_ds == 1:
    axes = [axes]

for ax, ds_name in zip(axes, DATASET_NAMES):
    sub = df[df['dataset'] == ds_name]

    for mi, mname in enumerate(model_names):
        vals = sub[sub['model'] == mname]['nll_full'].values
        jitter = np.random.uniform(-0.1, 0.1, len(vals))
        ax.scatter(mi + jitter, vals, color=COR[mname], s=80,
                   alpha=0.7, edgecolors='white', linewidths=0.5, zorder=3)
        ax.hlines(vals.mean(), mi - 0.2, mi + 0.2,
                  color=COR[mname], lw=3, zorder=4)

    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, fontsize=12)
    ax.set_ylabel('NLL full (nats)', fontsize=11)
    ax.set_title(f'{ds_name}', fontweight='bold', fontsize=13)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f'NLL de Extrapolacao por Seed\n'
    f'Treino: {TRAIN_LEN} eventos  |  Teste: sequencias completas  |  Linha = media',
    fontsize=13, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.savefig('extrapolation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 9 — Diagnostico: NLL de validacao por seed
# ============================================================================

fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 5))
if n_ds == 1:
    axes = [axes]

for ax, ds_name in zip(axes, DATASET_NAMES):
    sub = df[df['dataset'] == ds_name]

    for mi, mname in enumerate(model_names):
        vals = sub[sub['model'] == mname]['val_nll'].values
        x_pts = np.arange(len(vals))
        ax.scatter(x_pts + mi * 0.15, vals, color=COR[mname], s=90,
                   label=mname, edgecolors='white', linewidths=0.5, zorder=3)

    ax.set_xticks(np.arange(N_SEEDS))
    ax.set_xticklabels([f's{i}' for i in range(N_SEEDS)], fontsize=10)
    ax.set_xlabel('Seed')
    ax.set_ylabel('NLL de validacao')
    ax.set_title(f'{ds_name}', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('NLL de Validacao por Seed — Diagnostico de Convergencia',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('extrapolation_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 10 — Tabela para o paper
# ============================================================================

print('=' * 75)
print('TABELA PARA O PAPER — EXTRAPOLACAO EM DADOS REAIS')
print(f'Treino: primeiros {TRAIN_LEN} eventos  |  Teste: sequencias completas')
print('=' * 75)
print()

header = f'{"Dataset":>13s} {"Model":>6s}  {"NLL short":>18s}  {"NLL full (extrap)":>18s}'
print(header)
print(f'{"-"*13} {"-"*6}  {"-"*18}  {"-"*18}')

for ds_name in DATASET_NAMES:
    sub = df[df['dataset'] == ds_name]
    test_lens = [len(x['time_since_start']) for x in dataset_info[ds_name]['test_long']]
    mean_extrap = np.mean(test_lens) / TRAIN_LEN

    best_full = sub.groupby('model')['nll_full'].mean().idxmin()

    for mi, mname in enumerate(model_names):
        rows = sub[sub['model'] == mname]
        short_str = f'{rows["nll_short"].mean():.4f} +/- {rows["nll_short"].std():.4f}'
        full_str  = f'{rows["nll_full"].mean():.4f} +/- {rows["nll_full"].std():.4f}'

        marker = '*' if mname == best_full else ' '
        if mi == 0:
            ds_label = f'{ds_name} ({mean_extrap:.0f}x)'
        else:
            ds_label = ''
        print(f'{ds_label:>13s} {mname:>6s}   {short_str:>18s}  {marker}{full_str:>18s}')
    print()

print('* = melhor NLL na sequencia completa (cenario de extrapolacao)')
print(f'n = {N_SEEDS} seeds')

## Interpretacao

### O que estamos medindo?

- **NLL short:** NLL nos primeiros 10 eventos (mesma distribuicao do treino). Todos os modelos devem performar de forma similar aqui.
- **NLL full:** NLL na sequencia **inteira** (~50-90 eventos). O modelo precisa extrapolar para posicoes que nunca viu no treino.

A diferenca entre NLL full e NLL short mostra quanto o modelo **degrada** ao extrapolar.

### Por que o HoTHP deve ter NLL full menor?

O kernel hiperbolico do HoTHP eh **monotonicamente decrescente** com a distancia temporal.
A atencao aprendida em sequencias curtas se generaliza naturalmente para posicoes mais distantes.

O RoTHP (RoPE) e o THP usam kernels trigonometricos/sinusoidais que interpolam bem
dentro da faixa de treino, mas nao garantem monotonicidade para extrapolacao.